In [ ]:
%reload_ext dotenv
%dotenv

In [ ]:
import asyncio
from io import BytesIO
from urllib.parse import urlparse

from PIL import Image
from google import genai
from google.genai import types
from tenacity import retry, stop_after_attempt, wait_exponential
import replicate
import requests

In [ ]:
def preview(img, size=600):
    p = img.copy()
    p.thumbnail((size, size))
    display(p)

## VTON

In [ ]:
wearables = [
    {"path": "../images/garments/tops/sweater.jpg", "mask_prompt": "sweater", "category": "top"},
    {"path": "../images/garments/tops/striped_sweater.webp", "mask_prompt": "sweater", "category": "top"},
    {"path": "../images/garments/tops/sweater2.webp", "mask_prompt": "sweater", "category": "top"},
    {"path": "../images/garments/tops/tshirt.webp", "mask_prompt": "t-shirt", "category": "top"},
    {"path": "../images/garments/bottoms/gym_shorts.webp", "mask_prompt": "shorts", "category": "bottom"},
    {"path": "../images/garments/bottoms/jeans.webp", "mask_prompt": "pants", "category": "bottom"},
    {"path": "../images/garments/bottoms/joggers.jpg", "mask_prompt": "pants", "category": "bottom"},
]

avatar = Image.open("../images/avatars/nimo.jpg")
preview(avatar, size=400)
# for w in wearables:
#     image = Image.open(w["path"])
#     preview(image, size=400)

In [ ]:
client = genai.Client()


@retry(wait=wait_exponential(multiplier=1, min=2, max=30), stop=stop_after_attempt(3))
async def run_vton(avatar: Image.Image, wearable: Image.Image) -> Image.Image:
    prompt = """
    Put this garment on the avatar.
    Preserve the avatar's art style exactly.
    Keep the same pose, background, and all other clothing unchanged.
    """

    response = await client.aio.models.generate_content(
        model="gemini-3.1-flash-image-preview",
        contents=[avatar, wearable, prompt],
        config=types.GenerateContentConfig(
            image_config=types.ImageConfig(
                aspect_ratio="3:4",
                image_size="1K",
            ),
        ),
    )

    assert response.parts is not None

    result = None
    for part in response.parts:
        if part.text is not None:
            print(part.text)
        elif part.inline_data is not None:
            genai_image = part.as_image()
            assert genai_image is not None
            assert genai_image.image_bytes is not None
            result = Image.open(BytesIO(genai_image.image_bytes))

    usage = response.usage_metadata
    assert usage is not None
    print(
        f"Token usage: {usage.prompt_token_count} input, {usage.candidates_token_count} output, {usage.total_token_count} total"
    )

    assert result is not None
    return result


wearable_images = [Image.open(w["path"]) for w in wearables]
vton_results = await asyncio.gather(*(run_vton(avatar, w) for w in wearable_images))

for w, result in zip(wearables, vton_results):
    print(f"\n{w['path']}:")
    print(f"Image size: {result.size} px")
    preview(result)

## Masking

In [ ]:
@retry(wait=wait_exponential(multiplier=1, min=2, max=30), stop=stop_after_attempt(3))
async def generate_mask(woa_image: Image.Image, mask_prompt: str) -> Image.Image:
    woa_buf = BytesIO()
    woa_image.save(woa_buf, format="JPEG")
    woa_buf.seek(0)

    mask_results = await asyncio.to_thread(
        replicate.run,
        "schananas/grounded_sam:ee871c19efb1941f55f66a3d7d960428c8a5afcb77449547fe8e5a3ab9ebc21c",
        input={
            "image": woa_buf,
            "mask_prompt": mask_prompt,
            "negative_mask_prompt": "",
            "adjustment_factor": 0,
        },
    )

    mask_image_url = None
    for result_url_raw in mask_results:
        result_url_parsed = urlparse(str(result_url_raw))
        if result_url_parsed.path.endswith("/mask.jpg"):
            mask_image_url = result_url_parsed.geturl()
            break

    if mask_image_url is None:
        raise ValueError("Could not get mask URL from grounded_sam results")

    mask_response = requests.get(mask_image_url)
    mask_response.raise_for_status()
    return Image.open(BytesIO(mask_response.content))

masks = await asyncio.gather(*(
    generate_mask(vton_results[i], wearables[i]["mask_prompt"])
    for i in range(len(wearables))
))

In [ ]:
from IPython.display import display, HTML
import base64

def img_to_data_uri(img, size=400):
    p = img.copy()
    p.thumbnail((size, size))
    buf = BytesIO()
    p.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

for w, woa, mask in zip(wearables, vton_results, masks):
    # Mask (B&W)
    mask_resized = mask.resize(woa.size).convert("L")

    # Mask overlay on WOA (semi-transparent red)
    overlay = woa.copy().convert("RGBA")
    red = Image.new("RGBA", woa.size, (255, 0, 0, 100))
    overlay.paste(red, (0, 0), mask_resized)

    # Composite WOA onto avatar using mask
    composite = avatar.copy().resize(woa.size)
    woa_resized = woa.resize(composite.size)
    mask_for_paste = mask.resize(composite.size).convert("L")
    composite.paste(woa_resized, (0, 0), mask_for_paste)

    display(HTML(f"""
        <p><strong>{w['path']}</strong></p>
        <div style="display:flex;gap:8px">
            <div><img src="{img_to_data_uri(mask_resized)}" /><br><small>Mask</small></div>
            <div><img src="{img_to_data_uri(overlay)}" /><br><small>Overlay</small></div>
            <div><img src="{img_to_data_uri(composite)}" /><br><small>Composite</small></div>
        </div>
    """))